# Foundry IQ knowledge base with Fabric Data Agent integration

This notebook creates a Foundry IQ knowledge base that combines indexed company documents with a **Fabric Data Agent Knowledge Source**.

- **Fabric Data Agent Knowledge Source**: connects Azure AI Search to a published data agent in a Fabric workspace for structured and analytical questions.
- **Delegated token**: uses the signed-in user's identity when Azure AI Search queries the protected Fabric source.

The Fabric Data Agent knowledge source is currently exposed by the preview Azure AI Search SDK even though its public documentation is limited.

## Step 1: Set up Azure AI Search, Azure OpenAI, and Fabric configuration

Load the resource configuration from the project `.env` file. This notebook requires the ID of the published Fabric Data Agent created during provisioning.

In [1]:
import json
import os

from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY")  # optional; unset uses managed identity
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_DATA_AGENT_ID = os.environ["FABRIC_DATA_AGENT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"
HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME = "fabric-data-agent-knowledge-source"
KNOWLEDGE_BASE_NAME = "multisource-fabric-data-agent-knowledge-base"

search_credential = AzureDeveloperCliCredential(tenant_id=AZURE_TENANT_ID)
print("Environment variables loaded")

Environment variables loaded


## Step 2: Authenticate with a delegated user token

The retrieval call needs an OAuth token for a signed-in user who can access the Fabric workspace and published data agent. Azure AI Search forwards this delegated identity when querying the protected source.

Run the cell below to sign in and request the Azure AI Search scope used by `query_source_authorization`.

In [2]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

/Users/pamelafox/iqdeepdive-foundryiq-1/.venv/lib/python3.14/site-packages/msal/oauth2cli/oauth2.py:453: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


Acquired token for logged-in user


## Step 3: Create the search index knowledge sources

Create or update the same HR and health document sources used by the other multi-source labs.

- `hrdocs-knowledge-source`: points to the `hrdocs` search index
- `healthdocs-knowledge-source`: points to the `healthdocs` search index

In [3]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    credential=search_credential,
    api_version=AZURE_SEARCH_API_VERSION,
)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

print("Search index knowledge sources created")

Search index knowledge sources created


## Step 4: Create the Fabric Data Agent knowledge source

The `azure-search-documents` package exposes a Fabric Data Agent source that identifies the published agent by its Fabric workspace ID and data agent ID.

In [ ]:
from azure.search.documents.indexes.models import (
    FabricDataAgentKnowledgeSource,
    FabricDataAgentKnowledgeSourceParameters,
)

fabric_data_agent_knowledge_source = FabricDataAgentKnowledgeSource(
    name=FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
    description="Data agent with access to product inventory, reviews, and website analytics",
    fabric_data_agent_parameters=FabricDataAgentKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        data_agent_id=FABRIC_DATA_AGENT_ID,
    ),
)
index_client.create_or_update_knowledge_source(
    knowledge_source=fabric_data_agent_knowledge_source
)

print("Fabric Data Agent knowledge source created")

Fabric Data Agent knowledge source created


## Step 5: Create the multi-source knowledge base

Combine the two document indexes and the Fabric Data Agent with an Azure OpenAI chat model. Answer synthesis lets the knowledge base produce one response across the selected sources, while low reasoning effort enables source selection and query planning.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_source_names = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
 ]

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining company documents, product data, website analytics",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in knowledge_source_names],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions=(
        "Use the Fabric Data Agent for structured product, inventory, and analytical questions. "
        "Use the search indexes for HR and health policy documents."
    ),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

Knowledge base created


## Step 6: Query the knowledge base

Ask a question that requires current inventory analysis from the Fabric Data Agent and role information from the indexed HR documents. The data agent is always queried for this request; agentic retrieval can decide whether each document source is also needed.

Retrieval can take longer than an index-only query because Azure AI Search invokes the published Fabric Data Agent.

In [7]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricDataAgentKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
    api_version=AZURE_SEARCH_API_VERSION,
)

question = (
    "I'm a Contoso buyer preparing for the summer sale. "
    "Which products or product categories have the lowest or zero inventory right now, and where? "
    "Which Contoso job roles are responsible for inventory strategy and budget oversight?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricDataAgentKnowledgeSourceParams(
            knowledge_source_name=FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))

Here’s a buyer-focused summary of the current low/zero inventory picture and the Contoso roles most relevant to inventory strategy and budget oversight.

**Products/categories with zero inventory right now, and where**

The retrieved inventory summary identifies these items as having **zero inventory** at the listed store locations:[ref_id:0]

- **Hand Tools** include **Adjustable Wrench 10-inch** at **Kirkland**,[ref_id:0] **Metric Wrench Set** at **Bellevue** and **Spokane**,[ref_id:0] **Combination Wrench Set SAE** at **Seattle**,[ref_id:0] and **Flathead Screwdriver Set** at **Bellevue** and **Spokane**.[ref_id:0]

- **Lumber & Building Materials** has multiple zero-inventory items across **Tacoma, Seattle, Everett, Kirkland, Bellevue, Spokane, and Redmond**,[ref_id:0] including **Pressure Treated 2x4x8**,[ref_id:0] **Cedar Board 1x8x10**,[ref_id:0] **Douglas Fir 2x6x10**,[ref_id:0] **Deck Post 6x6x10**,[ref_id:0] **PT Landscape Timber**,[ref_id:0] **PVC Trim Board 1x6**,[ref_id:0] **Spray Foam Kit**,[ref_id:0] **Crown Molding Pine 8ft**,[ref_id:0] **WonderBoard Lite**,[ref_id:0] **OSB Subflooring 4x8x3/4**,[ref_id:0] **Marine Plywood 4x8x3/4**,[ref_id:0] and **Standard Drywall 4x8x1/2**.[ref_id:0]

- **Electrical** has zero inventory for **Retractable Cord Reel**,[ref_id:0] **Standard Outlet 15-Amp**,[ref_id:0] **Ceiling Box with Bracket**,[ref_id:0] **Track Lighting Kit**,[ref_id:0] **Pendant Light Kitchen**,[ref_id:0] **Outdoor Flood Light LED**,[ref_id:0] **Friction Tape**,[ref_id:0] **Heavy Duty Cord 100ft**,[ref_id:0] **PVC Conduit 3/4-inch**,[ref_id:0] **Romex Wire 14-2 250ft**,[ref_id:0] and **USB Outlet with Charging**,[ref_id:0] across **Kirkland, Redmond, Bellevue, Tacoma, and Seattle**.[ref_id:0]

- **Power Tools** has zero inventory for **Drywall Sander**,[ref_id:0] **File Belt Sander**,[ref_id:0] **Heavy Duty Jigsaw**,[ref_id:0] **Sliding Miter Saw 12-inch**,[ref_id:0] **Cordless Miter Saw 10-inch**,[ref_id:0] **Compound Miter Saw 10-inch**,[ref_id:0] **Compact Recip Saw**,[ref_id:0] **Reciprocating Saw Corded**,[ref_id:0] **Random Orbit Sander 5-inch**,[ref_id:0] and **Hammer Drill 1/2-inch**,[ref_id:0] across **Redmond, Bellevue, Everett, Tacoma, and Seattle**.[ref_id:0]

- **Paint & Finishes** has zero inventory for **Gloss Polyurethane**,[ref_id:0] **Clear Polyurethane Satin**,[ref_id:0] **Roller Frame 9-inch**,[ref_id:0] **Exterior Latex Paint Satin**,[ref_id:0] **Elastomeric Exterior Paint**,[ref_id:0] **Deck and Fence Stain**,[ref_id:0] and **Rust Prevention Spray**,[ref_id:0] across **Bellevue, Everett, Redmond, Tacoma, Spokane, and Seattle**.[ref_id:0]

- **Hardware** has zero inventory for **Deadbolt Lock Keyed**,[ref_id:0] **Butt Hinge 3-1/2 inch**,[ref_id:0] **Hex Bolt Kit Grade 5**,[ref_id:0] **Carriage Bolt Set**,[ref_id:0] **Countertop Support Bracket**,[ref_id:0] **Security Hasp 6-inch**,[ref_id:0] **Rubber Washer Set**,[ref_id:0] and **Swivel Caster 2-inch**,[ref_id:0] across **Kirkland, Everett, Spokane, Tacoma, Seattle, and Bellevue**.[ref_id:0]

- **Garden & Outdoor** has zero inventory for **Solar Path Light Set**,[ref_id:0] **Pole Saw Pruner 8-foot**,[ref_id:0] **Misting Sprinkler Kit**,[ref_id:0] **All-Purpose Fertilizer 50lb**,[ref_id:0] **Slow Release Granules**,[ref_id:0] **Soaker Hose 25-foot**,[ref_id:0] **LED Flood Light 30W**,[ref_id:0] and **Potting Mix Premium 40lb**,[ref_id:0] across **Seattle, Spokane, Tacoma, Bellevue, Kirkland, and Redmond**.[ref_id:0]

- **Storage & Organization** has zero inventory for **Rolling Tool Chest 26-inch**,[ref_id:0] **Chemical Storage Cabinet**,[ref_id:0] **Modular Cabinet System**,[ref_id:0] **Wall Mount Cabinet**,[ref_id:0] **Adjustable Height Workbench**,[ref_id:0] and **Safety Net for Storage**,[ref_id:0] across **Spokane, Redmond, Tacoma, and Bellevue**.[ref_id:0]

- **Plumbing** has zero inventory for **Fiberglass Pipe Insulation**,[ref_id:0] **Fill Valve Assembly**,[ref_id:0] **Kitchen Sink Faucet**,[ref_id:0] **Shower Faucet Trim Kit**,[ref_id:0] **Ball Valve 1/2-inch**,[ref_id:0] **PVC Tee Fitting Set**,[ref_id:0] **Compression Fittings**,[ref_id:0] **Push-Fit Connectors**,[ref_id:0] **Copper Pipe 1-inch Type L**,[ref_id:0] **Pre-Insulated Copper Pipe**,[ref_id:0] **Pipe Dope Stick**,[ref_id:0] **Check Valve 1-inch**,[ref_id:0] **Pipe Insulation 3/4-inch**,[ref_id:0] and **Toilet Seat Standard**,[ref_id:0] across **Everett, Redmond, Bellevue, Spokane, Kirkland, Seattle, and Tacoma**.[ref_id:0]

**Which areas appear most constrained for summer-sale readiness**

Based on the inventory summary, the broadest zero-inventory exposure appears in **Lumber & Building Materials** and **Plumbing**, because those categories show many zero-stock SKUs spread across the largest number of stores.[ref_id:0]

Several stores recur across many zero-inventory categories, especially **Seattle, Bellevue, Tacoma, Redmond, Spokane, Kirkland, and Everett**,[ref_id:0] which suggests those locations deserve priority review before the sale.[ref_id:0]

**Contoso job roles responsible for inventory strategy and budget oversight**

From the Contoso role documents, the clearest role tied directly to **inventory strategy** is the **Director of Operations**, because the role explicitly includes developing and implementing strategies to manage **inventory and stock levels**.[ref_id:1] The same role also **manages the budget and financial operations of the company**,[ref_id:1] so it spans both inventory planning and budget oversight.[ref_id:1]

The **Vice President of Operations** is also relevant for budget oversight and broader operating strategy, because the role is responsible for developing and implementing operational plans and **budgets** and for improving efficiency, reducing costs, and maximizing profitability.[ref_id:11]

The **Senior Manager of Operations** is relevant as well because the role oversees supply chain, logistics, and finance and is responsible for developing, implementing, and monitoring **operational plans, budgets, and performance goals**.[ref_id:9]

The **Manager of Operations** also has budget oversight responsibilities because the role includes developing and managing **operational budgets** and monitoring financial performance.[ref_id:18]

For explicit financial oversight, the **Chief Financial Officer** is responsible for Contoso’s financial operations, including **budgeting, financial planning and analysis, forecasting, financial reporting, and risk management**.[ref_id:3]

I did **not** find an HR role document explicitly titled **Inventory Planner**, **Inventory Analyst**, **Category Manager**, or **Buyer** in the retrieved Contoso role library results for this turn,[ref_id:1][ref_id:17] so the strongest document-backed answers are the operations and finance roles above.[ref_id:1][ref_id:3][ref_id:9][ref_id:11][ref_id:18]

**Practical short list for you to contact**

For immediate summer-sale action, the most document-supported contacts would be **Director of Operations** for inventory/stock strategy,[ref_id:1] **VP or Manager-level Operations** for execution and operating budgets,[ref_id:11][ref_id:18] and **CFO/finance leadership** for enterprise budget oversight.[ref_id:3]

If you want, I can next turn this into a **store-by-store priority table** or a **top categories to replenish first** list using the zero-inventory results above.[ref_id:0]

## Step 7: Inspect retrieval activity

Review the query plan and source operations. A successful mixed-source request should include a Fabric Data Agent activity record and any search-index queries selected by agentic retrieval.

In [8]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1534,
    "outputTokens": 120,
    "modelName": "gpt-5.4",
    "elapsedMs": 2082
  },
  {
    "type": "searchIndex",
    "id": 2,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-07-24T00:18:18.2918124Z",
    "count": 37,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "inventory strategy role budget oversight job responsibilities",
      "filter": null,
      "semanticConfigurationName": "semantic-configuration",
      "sourceDataFields": [
        {
          "name": "snippet"
        },
        {
          "name": "uid"
        },
        {
          "name": "snippet_parent_id"
        },
        {
          "name": "blob_path"
        }
      ],
      "searchFields": [
        {
          "name": "snippet"
        }
      ]
    }
  },
  {
    "type": "searchIndex",
    "id": 3,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-07-24T00:18:

## Step 8: Inspect search and Data Agent references

Print each reference as returned by the preview API. Compare the document citations with the Fabric Data Agent reference, synthesized response, and structured source data.

Do not assume the Data Agent payload uses the same fields as a `fabricOntology` reference; inspect its actual `type` and `sourceData` values.

In [9]:
references_json = [reference.as_dict() for reference in result.references or []]
print(json.dumps(references_json, indent=2))

[
  {
    "type": "fabricDataAgent",
    "id": "0",
    "activitySource": 1,
    "sourceData": {
      "fabricAnswer": "Here\u2019s a summary for your summer sale preparations at Contoso:\n\n**Products or Product Categories with Zero Inventory (by Store/Location):**\n\n- **HAND TOOLS:**\n  - Adjustable Wrench 10-inch (Kirkland)\n  - Metric Wrench Set (Bellevue, Spokane)\n  - Combination Wrench Set SAE (Seattle)\n  - Flathead Screwdriver Set (Bellevue, Spokane)\n\n- **LUMBER & BUILDING MATERIALS:**\n  - Pressure Treated 2x4x8, Cedar Board 1x8x10, Douglas Fir 2x6x10, Deck Post 6x6x10, PT Landscape Timber, PVC Trim Board 1x6, Spray Foam Kit, Crown Molding Pine 8ft, WonderBoard Lite, OSB Subflooring 4x8x3/4, Marine Plywood 4x8x3/4, Standard Drywall 4x8x1/2 (Tacoma, Seattle, Everett, Kirkland, Bellevue, Spokane, Redmond)\n\n- **ELECTRICAL:**\n  - Retractable Cord Reel, Standard Outlet 15-Amp, Ceiling Box with Bracket, Track Lighting Kit, Pendant Light Kitchen, Outdoor Flood Light LED, Frict